# Prática — Aula 1: SOA na Prática

**Arquitetura Orientada a Serviços (SOA) e Web Services · FIAP**
**Aula 1 — Fundamentos de SOA**

Este notebook não exige nenhuma instalação — roda direto no Google Colab, só com a biblioteca padrão do Python.

**O que vamos fazer:**

1. Simular, na mão, uma chamada de serviço no estilo **SOAP clássico** (envelope XML) e no estilo **REST moderno** (JSON).
2. Comparar, com números reais, o peso e a complexidade de cada abordagem.
3. Construir um **mini-ESB**: um roteador central com adaptadores que traduzem três sistemas heterogêneos — exatamente o cenário da **TransLog** discutido em aula — para um formato canônico único.
4. Testar o mini-ESB de ponta a ponta com um pedido de exemplo.
5. Responder, por escrito, questões de discussão conectando a prática aos conceitos da aula (os quatro *tenets* de Don Box e os princípios de Thomas Erl).

> Trabalhem em duplas. Leiam os comentários em cada célula antes de executar — eles fazem parte do material de estudo.

## Antes de Começar — Sua Missão

Este notebook tem **5 falhas escondidas** nas células de código abaixo. Nenhuma delas quebra a execução — o notebook roda do início ao fim sem erro. O problema é que, em pelo menos cinco pontos, o resultado impresso está **sutilmente errado**: um valor trocado, uma métrica que não mede o que diz medir, uma informação que desaparece silenciosamente.

**O que fazer:**

1. Executem o notebook célula por célula, prestando atenção real aos valores impressos — não só se rodou, mas se o resultado faz sentido.
2. Quando desconfiarem de algo, usem uma IA (Claude, ChatGPT, Copilot, o que preferirem) para ajudar a diagnosticar e corrigir — mas expliquem para a IA o que vocês observaram, não apenas colem o código inteiro pedindo "conserta isso".
3. Preencham o **Diário de Debugging** na Parte H, no final do notebook, documentando cada falha encontrada: onde estava, o que estava errado, como perceberam, e por que a correção é a certa — não basta a IA ter sugerido algo, vocês precisam entender e justificar.

Dica: os cinco bugs se conectam a conceitos vistos em aula — contratos malformados, benchmarks mal desenhados e agregação incompleta de dados. Se um resultado parecer bom demais (ou estranho demais) para ser verdade, provavelmente é um dos bugs.

## Parte A — O Cenário

Vamos usar o mesmo cenário do mini-case discutido em aula:

> A **TransLog** é uma empresa de logística com 15 anos de operação. Ela roda um **ERP legado (SAP)** para financeiro e estoque, um **sistema de rastreamento de entregas em Java exposto via SOAP/WSDL**, e acabou de adquirir uma **startup de roteamento de última milha** cuja plataforma é 100% baseada em **microsserviços REST/JSON**.

O time de TI precisa consultar o status de um pedido nos três sistemas e apresentar uma visão única para o cliente final. É exatamente isso que vamos construir.

## Parte B — Simulando uma Chamada SOAP Clássica

Antes do REST existir como padrão dominante, uma chamada típica de Web Service era feita através de uma mensagem **SOAP**: um envelope XML contendo cabeçalho e corpo, definido por um contrato **WSDL**.

Vamos construir, na mão, a mensagem de **requisição** e a mensagem de **resposta** para a operação `consultarPedido` do sistema legado de rastreamento da TransLog — exatamente como um desenvolvedor faria (ou como uma ferramenta geraria a partir do WSDL) no início dos anos 2000.

In [ ]:
import xml.etree.ElementTree as ET
import xml.dom.minidom as minidom
import time

def montar_requisicao_soap(pedido_id: str) -> str:
    """Monta uma mensagem SOAP de requisição para consultar um pedido."""
    envelope = ET.Element("soap:Envelope", {
        "xmlns:soap": "http://schemas.xmlsoap.org/soap/envelope/",
        "xmlns:tl": "http://translog.com.br/rastreamento",
    })
    header = ET.SubElement(envelope, "soap:Header")
    auth = ET.SubElement(header, "tl:AuthToken")
    auth.text = "legacy-token-9f21"
    body = ET.SubElement(envelope, "soap:Body")
    consulta = ET.SubElement(body, "tl:consultarPedido")
    id_el = ET.SubElement(consulta, "tl:pedidoId")
    id_el.text = pedido_id
    xml_bruto = ET.tostring(envelope, encoding="unicode")
    return minidom.parseString(xml_bruto).toprettyxml(indent="  ")

req_soap = montar_requisicao_soap("TL-48291")
print(req_soap)

<?xml version="1.0" ?>
<soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/" xmlns:tl="http://translog.com.br/rastreamento">
  <soap:Header>
    <tl:AuthToken>legacy-token-9f21</tl:AuthToken>
  </soap:Header>
  <soap:Body>
    <tl:consultarPedido>
      <tl:pedidoId>TL-48291</tl:pedidoId>
    </tl:consultarPedido>
  </soap:Body>
</soap:Envelope>



In [ ]:
def montar_resposta_soap(pedido_id: str, status: str, previsao: str) -> str:
    """Monta a mensagem SOAP de resposta que o sistema legado devolveria."""
    envelope = ET.Element("soap:Envelope", {
        "xmlns:soap": "http://schemas.xmlsoap.org/soap/envelope/",
        "xmlns:tl": "http://translog.com.br/rastreamento",
    })
    body = ET.SubElement(envelope, "soap:Body")
    resposta = ET.SubElement(body, "tl:consultarPedidoResponse")
    id_el = ET.SubElement(resposta, "tl:pedidoId")
    id_el.text = pedido_id
    status_el = ET.SubElement(resposta, "tl:status")
    status_el.text = previsao
    prev_el = ET.SubElement(resposta, "tl:previsaoEntrega")
    prev_el.text = status
    xml_bruto = ET.tostring(envelope, encoding="unicode")
    return minidom.parseString(xml_bruto).toprettyxml(indent="  ")

resp_soap = montar_resposta_soap("TL-48291", "2026-08-07", "EM_TRANSITO")
print(resp_soap)


<?xml version="1.0" ?>
<soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/" xmlns:tl="http://translog.com.br/rastreamento">
  <soap:Body>
    <tl:consultarPedidoResponse>
      <tl:pedidoId>TL-48291</tl:pedidoId>
      <tl:status>EM_TRANSITO</tl:status>
      <tl:previsaoEntrega>2026-08-07</tl:previsaoEntrega>
    </tl:consultarPedidoResponse>
  </soap:Body>
</soap:Envelope>



## Parte C — A Mesma Chamada em Estilo REST/JSON

Agora vamos representar a **mesma operação** — consultar o status de um pedido — do jeito que uma API REST moderna faria: um `GET` sobre um recurso, identificado por uma URL, retornando uma representação em JSON.

Repare que não existe "mensagem de requisição" no mesmo sentido do SOAP: a própria URL + o método HTTP **são** a requisição.

In [ ]:
import json

# Estilo REST: a requisição "é" a própria chamada HTTP.
# GET /pedidos/TL-48291  (não existe corpo de requisição para uma consulta simples)
url_rest = "GET /pedidos/TL-48291 HTTP/1.1"
print(url_rest)
print("Host: api.translog.com.br")
print("Authorization: Bearer eyJhbGciOi... (JWT — veremos em detalhe no 2º semestre)")


GET /pedidos/TL-48291 HTTP/1.1
Host: api.translog.com.br
Authorization: Bearer eyJhbGciOi... (JWT — veremos em detalhe no 2º semestre)


In [ ]:
def montar_resposta_rest(pedido_id: str, status: str, previsao: str) -> str:
    """Monta a resposta REST/JSON equivalente à resposta SOAP acima."""
    corpo = {
        "pedidoId": pedido_id,
        "status": status,
        "previsaoEntrega": previsao,
    }
    return json.dumps(corpo, indent=2, ensure_ascii=False)

resp_rest = montar_resposta_rest("TL-48291", "EM_TRANSITO", "2026-08-07")
print(resp_rest)


{
  "pedidoId": "TL-48291",
  "status": "EM_TRANSITO",
  "previsaoEntrega": "2026-08-07"
}


## Parte D — Comparando as Duas Abordagens

Na aula discutimos, em teoria, que mensagens SOAP são mais verbosas que JSON. Vamos comprovar isso com números — usando as próprias mensagens que construímos acima.

In [ ]:
tamanho_soap = len(resp_soap.encode("utf-8"))
tamanho_rest = len(resp_rest.encode("utf-8"))
razao = tamanho_soap / tamanho_rest

print(f"Tamanho da resposta SOAP (XML): {tamanho_soap} bytes")
print(f"Tamanho da resposta REST (JSON): {tamanho_rest} bytes")
print(f"A mensagem SOAP é {razao:.1f}x maior, para carregar exatamente a mesma informação de negócio.")


Tamanho da resposta SOAP (XML): 395 bytes
Tamanho da resposta REST (JSON): 90 bytes
A mensagem SOAP é 4.4x maior, para carregar exatamente a mesma informação de negócio.


In [ ]:
# Um teste rápido (e didático, não um benchmark rigoroso) de custo de parsing.
# Isso ilustra por que "overhead de XML" não é só sobre bytes na rede —
# também é sobre tempo de CPU gasto interpretando a mensagem.

N = 20000

inicio = time.perf_counter()
for _ in range(N):
    ET.fromstring(resp_soap)
tempo_xml = time.perf_counter() - inicio

resposta_json_pronta = json.loads(resp_rest)  # parseado uma vez, fora do loop

inicio = time.perf_counter()
for _ in range(N):
    resposta_json_pronta
tempo_json = time.perf_counter() - inicio

print(f"Tempo para fazer parsing {N} vezes:")
print(f"  XML (SOAP):  {tempo_xml:.4f} s")
print(f"  JSON (REST): {tempo_json:.4f} s")
print(f"  XML foi {tempo_xml / tempo_json:.1f}x mais lento para fazer parsing neste teste.")


Tempo para fazer parsing 20000 vezes:
  XML (SOAP):  0.1568 s
  JSON (REST): 0.0004 s
  XML foi 354.2x mais lento para fazer parsing neste teste.


**Pergunta para a dupla (respondam na Parte G):** os números acima parecem pequenos para uma única mensagem. Por que, mesmo assim, essa diferença é citada como um dos motivos reais para a migração de SOAP para REST em sistemas de grande escala? Pensem em volume — o V de Volume que vimos no curso de Big Data, se vocês já cursaram, ou simplesmente: o que acontece quando não é uma mensagem, e sim alguns milhões por dia?

## Parte E — Construindo um Mini-ESB

Agora a parte principal: vamos construir, em miniatura, o que um **Enterprise Service Bus** faz de verdade — um roteador central com um **adaptador** para cada sistema heterogêneo, todos traduzindo para um **formato canônico** único.

Vamos simular os três sistemas da TransLog:

- `sistema_sap` — o ERP legado, aqui simulado como um dicionário Python simples, com nomes de campo em português e nas convenções internas do SAP.
- `sistema_rastreamento_soap` — o sistema Java legado, que só "fala" SOAP/XML (reaproveitando exatamente as funções da Parte B).
- `sistema_startup_rest` — a plataforma da startup recém-adquirida, que só "fala" REST/JSON (reaproveitando a Parte C).

Cada um tem um formato de dado diferente. O trabalho do ESB é esconder essa diferença do consumidor final.

In [ ]:
# --- Simulação dos três sistemas heterogêneos (cada um só "fala" seu próprio formato) ---

def sistema_sap(pedido_id: str) -> dict:
    """Simula o ERP SAP: campos em português, convenção própria da TransLog."""
    return {
        "COD_PEDIDO": pedido_id,
        "VLR_TOTAL": 1249.90,
        "STATUS_FINANCEIRO": "PAGO",
    }

def sistema_rastreamento_soap(pedido_id: str) -> str:
    """Simula o sistema legado de rastreamento: só responde em SOAP/XML."""
    return montar_resposta_soap(pedido_id, status="EM_TRANSITO", previsao="2026-08-07")

def sistema_startup_rest(pedido_id: str) -> str:
    """Simula a API da startup de última milha: só responde em REST/JSON."""
    corpo = {
        "orderId": pedido_id,
        "lastMileCarrier": "TransLog Última Milha",
        "etaHours": 36,
    }
    return json.dumps(corpo, ensure_ascii=False)

print(sistema_sap("TL-48291"))
print(sistema_rastreamento_soap("TL-48291"))
print(sistema_startup_rest("TL-48291"))


{'COD_PEDIDO': 'TL-48291', 'VLR_TOTAL': 1249.9, 'STATUS_FINANCEIRO': 'PAGO'}
<?xml version="1.0" ?>
<soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/" xmlns:tl="http://translog.com.br/rastreamento">
  <soap:Body>
    <tl:consultarPedidoResponse>
      <tl:pedidoId>TL-48291</tl:pedidoId>
      <tl:status>2026-08-07</tl:status>
      <tl:previsaoEntrega>EM_TRANSITO</tl:previsaoEntrega>
    </tl:consultarPedidoResponse>
  </soap:Body>
</soap:Envelope>

{"orderId": "TL-48291", "lastMileCarrier": "TransLog Última Milha", "etaHours": 36}


In [ ]:
# --- Adaptadores: cada um traduz o formato nativo do seu sistema para o formato canônico do ESB ---

def adaptador_sap(pedido_id: str) -> dict:
    dados = sistema_sap(pedido_id)
    return {
        "origem": "SAP",
        "pedidoId": dados["COD_PEDIDO"],
        "statusFinanceiro": dados["VLR_TOTAL"],
        "valorTotal": dados["STATUS_FINANCEIRO"],
    }

def adaptador_rastreamento(pedido_id: str) -> dict:
    xml_resposta = sistema_rastreamento_soap(pedido_id)
    raiz = ET.fromstring(xml_resposta)
    ns = {"tl": "http://translog.com.br/rastreamento"}
    status = raiz.find(".//tl:status", ns).text
    previsao = raiz.find(".//tl:previsaoEntrega", ns).text
    return {
        "origem": "Rastreamento (SOAP legado)",
        "pedidoId": pedido_id,
        "statusEntrega": status,
        "previsaoEntrega": previsao,
    }

def adaptador_ultima_milha(pedido_id: str) -> dict:
    json_resposta = sistema_startup_rest(pedido_id)
    dados = json.loads(json_resposta)
    return {
        "origem": "Última Milha (REST)",
        "pedidoId": dados["orderId"],
        "transportadora": dados["etaHours"],
        "previsaoHoras": dados["lastMileCarrier"],
    }

# Teste rápido de cada adaptador isoladamente
for adaptador in (adaptador_sap, adaptador_rastreamento, adaptador_ultima_milha):
    print(adaptador("TL-48291"))


{'origem': 'SAP', 'pedidoId': 'TL-48291', 'statusFinanceiro': 1249.9, 'valorTotal': 'PAGO'}
{'origem': 'Rastreamento (SOAP legado)', 'pedidoId': 'TL-48291', 'statusEntrega': '2026-08-07', 'previsaoEntrega': 'EM_TRANSITO'}
{'origem': 'Última Milha (REST)', 'pedidoId': 'TL-48291', 'transportadora': 36, 'previsaoHoras': 'TransLog Última Milha'}


In [ ]:
# --- O ESB em si: orquestra os três adaptadores e monta uma visão única do pedido ---

def esb_consultar_pedido(pedido_id: str) -> dict:
    """
    Mini-ESB: chama os três sistemas através de seus adaptadores e combina
    o resultado em um único documento canônico, pronto para o app do cliente final.
    """
    financeiro = adaptador_sap(pedido_id)
    rastreamento = adaptador_rastreamento(pedido_id)
    ultima_milha = adaptador_ultima_milha(pedido_id)

    return {
        "pedidoId": pedido_id,
        "statusFinanceiro": financeiro["statusFinanceiro"],
        "valorTotal": financeiro["valorTotal"],
        "statusEntrega": rastreamento["statusEntrega"],
        "previsaoEntregaPrincipal": rastreamento["previsaoEntrega"],
        "ultimaMilha": {
            "transportadora": ultima_milha["transportadora"],
            "previsaoHoras": ultima_milha["previsaoHoras"],
        },
        "fontes": [financeiro["origem"], rastreamento["origem"]],
    }


## Parte F — Testando o Mini-ESB de Ponta a Ponta

Vamos chamar o ESB para um pedido e ver a visão unificada que ele entrega — exatamente o que o app do cliente final da TransLog mostraria, sem que o app precise saber que, por trás, existem três sistemas completamente diferentes.

In [ ]:
resultado = esb_consultar_pedido("TL-48291")
print(json.dumps(resultado, indent=2, ensure_ascii=False))


{
  "pedidoId": "TL-48291",
  "statusFinanceiro": 1249.9,
  "valorTotal": "PAGO",
  "statusEntrega": "2026-08-07",
  "previsaoEntregaPrincipal": "EM_TRANSITO",
  "ultimaMilha": {
    "transportadora": 36,
    "previsaoHoras": "TransLog Última Milha"
  },
  "fontes": [
    "SAP",
    "Rastreamento (SOAP legado)"
  ]
}


**Exercício da dupla:** adicionem um **quarto sistema** fictício — por exemplo, um sistema de avaliação do cliente (`sistema_avaliacao`), que só existe internamente como uma lista de tuplas `(pedido_id, nota)`. Escrevam:

1. A função que simula esse sistema.
2. O adaptador correspondente, traduzindo para o formato canônico.
3. Uma nova versão de `esb_consultar_pedido` que também incorpora essa informação.

Usem a célula abaixo.

In [ ]:
# Escreva aqui a simulação do quarto sistema, o adaptador e o ESB atualizado.



## Parte G — Diário de Debate (para entregar)

Respondam, em texto, nesta célula (ou em uma célula de markdown própria de cada integrante), conectando o que construíram acima aos conceitos da aula:

1. **Fronteiras explícitas.** No código acima, em que ponto exato o programa "cruza a fronteira" de um serviço? O que aconteceria, na vida real, que não acontece na nossa simulação, quando essa fronteira é cruzada (pensem nas Falácias da Computação Distribuída, se já viram o curso de Big Data, ou simplesmente: o que pode dar errado numa chamada de rede de verdade)?

2. **Contrato, não implementação.** Cada adaptador esconde um formato de dado bem diferente (dict, XML, JSON) atrás da mesma interface de saída. Qual princípio da orientação a serviços isso ilustra? O que aconteceria com o resto do sistema se o SAP mudasse o nome do campo `STATUS_FINANCEIRO` internamente?

3. **O ESB como gargalo.** Na Parte F, o ESB depende dos três sistemas respondendo. O que acontece com `esb_consultar_pedido` se `sistema_rastreamento_soap` estiver fora do ar? Isso ilustra qual crítica à SOA clássica que vimos em aula?

4. **Gancho para REST.** Comparando a Parte B com a Parte C: qual das duas abordagens vocês achraram mais rápida de escrever e entender? Por quê isso ajuda a explicar a migração da indústria para REST, mesmo os princípios de orientação a serviços continuando os mesmos?

> Registrem, se usaram alguma IA para ajudar a entender ou depurar o exercício desta prática, no que ela ajudou e por que a explicação fez sentido — igual combinamos para os notebooks de todas as disciplinas.

## Parte H — Diário de Debugging (para entregar)

Para **cada** uma das falhas que encontrarem, preencham um bloco como o modelo abaixo (copiem e repitam). Não precisam encontrar exatamente 5 — documentem quantas encontrarem, mas o notebook original tem 5.

---

**Falha #___**

- **Onde estava:** (nome da função/célula)
- **O que a célula deveria fazer:**
- **O que ela fazia de errado:**
- **Como perceberam:** (o que no output chamou atenção)
- **Como a IA ajudou a diagnosticar:** (o que vocês perguntaram, o que ela sugeriu)
- **A correção:** (trecho de código corrigido)
- **Por que essa correção é a certa** (não só "a IA disse" — expliquem com suas palavras):

---

**Falha #1**

- Onde estava: "montar_resposta_soap", o que precisa trocar de lugar é status_el.text = previsao e prev_el.text = status pois estão trocados e ta imprimindo status no lugar de previsão, e vice-versa.
- O que deveria fazer: precisa trocar de lugar é status_el.text = previsao e prev_el.text = status
- O que fazia de errado: estão trocados e ta imprimindo status no lugar de previsão, e vice-versa.
- Como perceberam: no print
- Como a IA ajudou: na parte conceitual: esse bug se conecta direto com o tenet de Don Box "contrato, não implementação" — o nome do campo no XML (<tl:status>) é parte do contrato público que o consumidor confia.
- Correção:
- Por que está certa:

**Falha #2**

- Onde estava:
- O que deveria fazer:
- O que fazia de errado:
- Como perceberam:
- Como a IA ajudou:
- Correção:
- Por que está certa:

**Falha #3**

- Onde estava:
- O que deveria fazer:
- O que fazia de errado:
- Como perceberam:
- Como a IA ajudou:
- Correção:
- Por que está certa:

**Falha #4**

- Onde estava:
- O que deveria fazer:
- O que fazia de errado:
- Como perceberam:
- Como a IA ajudou:
- Correção:
- Por que está certa:

**Falha #5**

- Onde estava:
- O que deveria fazer:
- O que fazia de errado:
- Como perceberam:
- Como a IA ajudou:
- Correção:
- Por que está certa: